# House Prices — разведочный анализ

Что смотрим:

1. что вообще в данных и где пропуски
2. распределение цены и почему её логарифмируют
3. как цена связана с качеством, площадью, возрастом и районом
4. выбросы
5. что дают признаки из `src/features.py`
6. быстрый baseline, чтобы было с чем сравнивать

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# чтобы работали импорты из src/ независимо от того, откуда запущен ноутбук
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 90)
plt.rcParams["figure.figsize"] = (10, 4)

train = pd.read_csv(ROOT / "data" / "raw" / "train.csv")
test = pd.read_csv(ROOT / "data" / "raw" / "test.csv")
target = "SalePrice"
print(train.shape, test.shape)
train.head()

## 1. Что в данных

In [ ]:
numeric = train.select_dtypes("number").columns.drop(["Id", target])
categorical = train.columns.difference(numeric).drop(["Id", target])
print(f"числовых колонок: {len(numeric)}, категориальных: {len(categorical)}")
train[numeric].describe().T

### Пропуски

Пропусков много, но почти все они не «неизвестно», а «этого нет»: `PoolQC` пуст,
когда нет бассейна, `GarageType` — когда нет гаража, и так далее (это написано в
`data_description.txt`). Поэтому в `src/features.py` оценки качества при пропуске
становятся 0, а категории — отдельным значением `missing`. По-настоящему неизвестен
в основном `LotFrontage`.

In [ ]:
missing = pd.DataFrame(
    {
        "train_%": train.isna().mean().mul(100).round(1),
        "test_%": test.isna().mean().mul(100).round(1),
    }
).query("`train_%` > 0 or `test_%` > 0")
missing.sort_values("test_%", ascending=False)

В тесте пропуски есть в колонках, где в train их не было (`MSZoning`, `BsmtFullBath`,
`GarageCars`, ...): по одной-две строки. Поэтому препроцессор заполняет пропуски
во всех колонках, а не только в «известных».

## 2. Целевая переменная

In [ ]:
print(train[target].describe().round(0))
print(f"\nскошенность: {train[target].skew():.2f}, после log1p: {np.log1p(train[target]).skew():.2f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].hist(train[target], bins=50)
axes[0].set_title("SalePrice")
axes[1].hist(np.log1p(train[target]), bins=50)
axes[1].set_title("log(1 + SalePrice)")
plt.tight_layout()
plt.show()

Цена сильно скошена вправо: немного очень дорогих домов. После логарифма
распределение почти симметричное. Метрика Kaggle — RMSE между логарифмами цен,
поэтому модель и учится на `log1p(SalePrice)` (`target_log: true` в конфиге):
ошибка в 10% на дешёвом и дорогом доме весит одинаково.

## 3. От чего зависит цена

In [ ]:
log_price = np.log1p(train[target])
corr = train[numeric].corrwith(log_price).sort_values(ascending=False)
pd.concat([corr.head(12), corr.tail(5)]).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

train.groupby("OverallQual")[target].median().plot(kind="bar", ax=axes[0], rot=0)
axes[0].set_title("Медианная цена по OverallQual")

axes[1].scatter(train["GrLivArea"], train[target], s=8, alpha=0.5)
axes[1].set_title("Цена от жилой площади")
axes[1].set_xlabel("GrLivArea")

axes[2].scatter(train["YearBuilt"], train[target], s=8, alpha=0.5)
axes[2].set_title("Цена от года постройки")
axes[2].set_xlabel("YearBuilt")

plt.tight_layout()
plt.show()

`OverallQual` — самый сильный одиночный признак, и зависимость от него явно
нелинейная (каждая следующая ступень дороже предыдущей) — в логарифмах она
выпрямляется. Площадь идёт второй. Справа на графике площади видны точки,
которые выбиваются из общего облака, — о них ниже.

In [ ]:
by_hood = train.groupby("Neighborhood")[target].agg(["median", "count"]).sort_values("median")
by_hood["median"].plot(kind="barh", figsize=(8, 7), title="Медианная цена по району")
plt.axvline(train[target].median(), color="gray", linestyle="--", linewidth=1)
plt.tight_layout()
plt.show()

Разница между районами — в разы. Район — категориальный признак с 25 значениями:
деревья справляются с ним через ordinal-кодирование, линейные модели — через onehot.

## 4. Выбросы

In [ ]:
huge = train["GrLivArea"] > 4000
train.loc[huge, ["Id", "GrLivArea", "OverallQual", "Neighborhood", "SaleCondition", target]]

Четыре дома больше 4000 кв. футов. Два из них стоят как средний дом — это частичные
продажи (`SaleCondition=Partial`) в одном районе. Автор датасета сам советует
убирать дома больше 4000 кв. футов. Мы убираем только эти два
(`data.drop_outliers: true`) и только из train.

Для линейной модели это критично: без удаления lasso даёт 0.1425 вместо 0.1094
(см. `RESULTS.md`). В тесте есть похожий дом (`Id=2550`) — на нём модель, скорее
всего, ошибётся, и с этим ничего не поделать.

## 5. Признаки из src/features.py

In [ ]:
from src.features import add_features

featured = add_features(train.drop(columns=["Id"]))
new_columns = sorted(set(featured.columns) - set(train.columns))
print("новые колонки:", new_columns)
featured[new_columns + ["OverallQual", "ExterQual", "KitchenQual"]].head()

In [ ]:
featured_numeric = featured.select_dtypes("number").drop(columns=[target])
featured_corr = featured_numeric.corrwith(np.log1p(featured[target])).sort_values(ascending=False)
featured_corr.head(15).round(3)

На что смотреть: `TotalSF` (общая площадь с подвалом) и `QualLivArea`
(качество × площадь) почти догоняют `OverallQual` и заметно сильнее любой
исходной площади (`GrLivArea` ~0.73). Оценки качества
(`ExterQual`, `KitchenQual`, ...) после перевода в числа 0–5 тоже попадают в верх
таблицы — в исходном виде это были строки, и корреляцию по ним не посчитать.

Корреляция линейная, так что низкое значение не значит «признак бесполезен».

## 6. Baseline

In [ ]:
import yaml

from src.training import train as run_training

with open(ROOT / "config.yaml", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

cfg["model"]["name"] = "lasso"
cfg["features"]["encoding"] = "onehot"
cfg["features"]["scale"] = True
cfg["output_dir"] = str(ROOT / "outputs")
cfg["data"]["train_path"] = str(ROOT / "data" / "raw" / "train.csv")
cfg["data"]["test_path"] = str(ROOT / "data" / "raw" / "test.csv")

metrics = run_training(cfg)
metrics

Дальше эксперименты удобнее гонять из терминала — там же они и логируются:

```bash
python main.py train --set model.name=lgbm
python main.py train --set model.name=ridge --set features.encoding=onehot --set features.scale=true
```

Результаты сравнений записывайте в `RESULTS.md`.

## Выводы

Заполните по итогам своего анализа — например:

- цена определяется в первую очередь качеством, площадью и районом;
- логарифм цены обязателен — и из-за скошенности, и из-за метрики;
- два выброса в train стоит удалить, иначе линейные модели разваливаются;
- что попробовать дальше: ...